# Session 3 — Track B: McKinsey Multi-Agent Engagement Team (Instructor Capstone)

In this capstone, students build a **multi-agent consulting engagement system** that decomposes complex client questions, delegates workstreams to specialized consulting agents, and orchestrates their collaboration using LangGraph — mirroring how a McKinsey engagement team tackles real client problems.

> **Instructor Note:** This notebook contains fully solved versions of all 4 milestones with detailed approach explanations. The self-correction loop (Milestone 4) is the most challenging — allow extra time for it.

**Engagement Scenario:** A Private Equity client is evaluating an acquisition target in the healthcare technology space. The engagement team must deliver a comprehensive due diligence assessment covering market dynamics, financial performance, competitive positioning, and operational readiness.

## Capstone Objectives

By the end of this capstone, students will have built:

1. An **Engagement Manager (supervisor) agent** that decomposes client questions into workstreams and assigns them to the right specialists
2. **Specialized consulting agents** — Strategy Analyst, Financial Analyst, Operations Expert, and Industry Researcher — each with distinct tools and expertise
3. A **LangGraph orchestration** layer managing workstream coordination, dependency sequencing, and insight synthesis
4. An **evaluation and self-correction** loop ensuring deliverable quality meets McKinsey standards

This mirrors the structure of a real McKinsey engagement: a Partner or EM coordinates workstreams, specialized analysts execute their workstreams, and the team synthesizes findings into a unified recommendation.

In [8]:
import os
from datetime import datetime
from collections import defaultdict
import numpy as np
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import time
from typing import TypedDict, Annotated, List, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
import operator

print("All imports successful!")

All imports successful!


---
## Milestone 1: Engagement Manager Agent & Workstream Decomposition

Build an Engagement Manager agent that decomposes complex client questions into workstreams and assigns them to the right consulting specialists.

> **Approach:** The Engagement Manager uses a structured prompt that instructs the LLM to return JSON with workstream decomposition. Each workstream includes an ID, hypothesis to test, assigned specialist, and dependencies on other workstreams. This enables dependency-aware execution in Milestone 3 — just as in a real engagement, some analyses must complete before others can begin (e.g., market sizing before competitive positioning).
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 15-20 minutes. This agent takes a complex question and decomposes it into workstreams. Emphasize that the LLM should return structured JSON with workstream details. Suggest testing with a simple question first ('Should we enter the Indian market?') before complex PE scenarios. The key quality measure: are the workstreams MECE and do they cover the question comprehensively?
>
> **Common Student Mistakes:**
> - The LLM not returning valid JSON -- use `response_format={'type': 'json_object'}` or add explicit JSON instructions
> - Workstreams that overlap (not MECE) -- instruct the LLM to make them mutually exclusive
> - Too many workstreams (5+) -- each one requires a specialist agent, so 2-4 is ideal for class time
> - Not including dependencies between workstreams -- some naturally depend on others' outputs
> - Hardcoding workstreams instead of letting the LLM decompose dynamically based on the question
>
> **Skippable?** NO -- CRITICAL -- this is the entry point for the entire multi-agent system. If the EM agent doesn't produce good workstreams, the specialist agents have nothing to work on. Do NOT skip.

In [9]:
# Milestone 1 — SOLUTION: Engagement Manager Agent & Workstream Decomposition

class EngagementManagerAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Engagement Manager leading a consulting engagement.
Decompose the client's question into 2-4 workstreams, each with a clear hypothesis to test.

Available specialists on your engagement team:
- strategy_analyst: Conducts market sizing, competitive analysis, and strategic positioning assessments
- financial_analyst: Performs financial modeling, valuation analysis, revenue/cost decomposition, and ROI calculations
- operations_expert: Evaluates operational readiness, process efficiency, org structure, and implementation feasibility
- industry_researcher: Researches industry trends, regulatory landscape, technology disruptions, and market dynamics

Return ONLY valid JSON in this format:
{
  "workstreams": [
    {"id": 1, "description": "...", "hypothesis": "...", "assigned_specialist": "industry_researcher", "dependencies": []},
    {"id": 2, "description": "...", "hypothesis": "...", "assigned_specialist": "strategy_analyst", "dependencies": [1]}
  ]
}"""

    def decompose(self, client_question: str) -> Dict:
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=client_question)
        ])
        try:
            plan = json.loads(response.content)
        except:
            # Fallback: simple two-workstream plan
            plan = {"workstreams": [
                {"id": 1, "description": f"Market & industry research: {client_question}", "hypothesis": "The market is attractive and growing", "assigned_specialist": "industry_researcher", "dependencies": []},
                {"id": 2, "description": f"Strategic assessment: {client_question}", "hypothesis": "The target has a defensible competitive position", "assigned_specialist": "strategy_analyst", "dependencies": [1]}
            ]}
        plan["client_question"] = client_question
        return plan


# Test — PE Due Diligence scenario
em = EngagementManagerAgent()
plan = em.decompose(
    "Our PE client is considering acquiring a mid-market healthcare IT company. "
    "Assess the target's market position, financial health, operational scalability, "
    "and provide a go/no-go recommendation with key risks and value creation levers."
)
print(json.dumps(plan, indent=2))

{
  "workstreams": [
    {
      "id": 1,
      "description": "Assess the target company's market position and competitive landscape within the healthcare IT sector.",
      "hypothesis": "The target company holds a strong competitive position in a growing healthcare IT market, which can be leveraged for future growth.",
      "assigned_specialist": "industry_researcher",
      "dependencies": []
    },
    {
      "id": 2,
      "description": "Evaluate the financial health of the target company through detailed financial modeling and valuation analysis.",
      "hypothesis": "The target company has a solid financial foundation with sustainable revenue streams and manageable costs.",
      "assigned_specialist": "financial_analyst",
      "dependencies": [
        1
      ]
    },
    {
      "id": 3,
      "description": "Analyze the operational scalability of the target company, focusing on process efficiency and organizational structure.",
      "hypothesis": "The target company h

---
## Milestone 2: Specialized Consulting Agents

Build specialized consulting agents, each with unique system prompts and analytical capabilities. These mirror the roles on a real McKinsey engagement team.

> **Approach:** Each specialist has a carefully crafted system prompt that defines their consulting expertise, analytical frameworks, and output format. Specialists receive both their workstream description and context from previously completed workstreams, allowing them to build on each other's insights — just as analysts on an engagement share findings across workstreams.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 15-20 minutes. Students need to implement 4 agent classes with distinct personas. Suggest a pattern: copy one agent class, then customize the system prompt and output format for each specialist. Emphasize that each agent should receive context from completed workstreams so they can build on each other's findings. If time is short, students can implement 2 agents and hardcode the other 2.
>
> **Common Student Mistakes:**
> - All agents using the same generic system prompt -- they should have distinct expertise and frameworks
> - Not passing cross-workstream context to agents -- they can't reference other agents' findings
> - Agent output format varying wildly -- standardize on a common structure (result text + metadata dict)
> - System prompts that are too long (500+ words) -- keep them focused on the agent's specific expertise
> - Not mapping specialist names to agent classes -- the dispatcher won't know which agent to invoke
>
> **Skippable?** NO -- CRITICAL -- the specialist agents are the workers in the multi-agent system. Without them, the orchestration (Milestone 3) has nothing to run. If pressed for time, implement 2 agents fully and stub the other 2 with simple LLM calls.

In [10]:
# Milestone 2 — SOLUTION: Specialized Consulting Agents

class StrategyAnalystAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Strategy Analyst specializing in competitive analysis and market positioning.
Given a workstream assignment:
1. Define the relevant market and key segments
2. Identify the competitive landscape (key players, market shares, positioning)
3. Assess the target's competitive advantages and vulnerabilities
4. Develop a hypothesis on strategic positioning with supporting evidence
5. Use frameworks like Porter's Five Forces, value chain analysis, or growth-share matrix where relevant

Structure your output with clear headers, bullet points, and a summary of key insights.
Use precise consulting language: hypotheses, implications, so-whats, and recommendations."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "strategy_analyst"}


class FinancialAnalystAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Financial Analyst specializing in financial modeling and valuation.
Given a workstream assignment:
1. Analyze revenue drivers and cost structure (decompose into fixed/variable, direct/indirect)
2. Assess margin trajectory and unit economics
3. Evaluate cash flow generation and working capital efficiency
4. Identify key financial risks and value creation levers
5. Provide a preliminary view on valuation (multiples-based or DCF framework)

Structure your output with clear financial metrics, benchmarks against industry peers, and actionable insights.
Quantify everything — use ranges where exact figures are unavailable."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "financial_analyst"}


class OperationsExpertAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Operations Expert specializing in operational assessment and transformation.
Given a workstream assignment:
1. Evaluate the operating model (org structure, processes, technology stack)
2. Assess scalability — can the organization grow 2-3x without breaking?
3. Identify operational risks and bottlenecks
4. Propose efficiency improvements and implementation roadmap
5. Estimate the effort, timeline, and investment required for key initiatives

Structure your output with clear categories, maturity assessments, and prioritized recommendations.
Use frameworks like operating model canvas, lean assessment, or capability maturity models."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "operations_expert"}


class IndustryResearcherAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are a McKinsey Industry Researcher specializing in sector-specific knowledge and market intelligence.
Given a workstream assignment:
1. Map the industry landscape: market size, growth rate, key trends
2. Identify regulatory factors, compliance requirements, and policy tailwinds/headwinds
3. Assess technology disruptions and their impact on the sector
4. Analyze customer segments, buying behaviors, and unmet needs
5. Provide forward-looking perspective: where is this industry heading in 3-5 years?

Structure your output with data-driven insights, sourced estimates where possible, and clear implications.
Be rigorous in separating facts from hypotheses."""
    
    def execute(self, workstream, context=""):
        prompt = f"Workstream: {workstream}"
        if context:
            prompt += f"\n\nInsights from other workstreams:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "industry_researcher"}


# Test — Strategy Analyst on a competitive analysis workstream
strategy_analyst = StrategyAnalystAgent()
result = strategy_analyst.execute(
    "Assess the competitive positioning of the target healthcare IT company. "
    "Identify top 3-5 competitors, relative market share, and key differentiators."
)
print(f"Specialist: {result['worker']}")
print(f"Analysis (preview): {result['result'][:400]}...")

Specialist: strategy_analyst
Analysis (preview): # Competitive Positioning Assessment of Target Healthcare IT Company

## 1. Relevant Market and Key Segments
### Market Definition
- **Healthcare IT Market**: Encompasses software, hardware, and services that facilitate the management of healthcare data, improve patient care, and streamline operations within healthcare organizations.
  
### Key Segments
- **Electronic Health Records (EHR) Systems*...


---
## Milestone 3: LangGraph Engagement Orchestration

Build a LangGraph workflow that orchestrates the Engagement Manager and consulting specialists into a coordinated engagement team.

> **Approach:** We define a StateGraph with an Engagement Manager node (creates the workstream plan), a dispatcher node (picks the next workstream and routes it to the right specialist), specialist nodes (execute workstreams), and a synthesis node (combines all workstream insights into a unified client deliverable). The dispatcher uses conditional routing to assign each workstream to the correct specialist, and loops back for more workstreams until the full engagement plan is executed — mirroring how an EM coordinates a real engagement.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 20-25 minutes -- this is the most complex milestone. Draw the full graph on the board: EM -> Dispatcher -> Specialist -> (loop back to Dispatcher if more workstreams) -> Synthesis. The dispatcher node is the key: it picks the next workstream and routes to the right specialist. Suggest getting the EM -> Dispatcher -> one Specialist -> Synthesis linear path working first, then adding the loop.
>
> **Common Student Mistakes:**
> - The dispatcher not correctly tracking `current_workstream_idx` -- it processes the same workstream repeatedly
> - Conditional routing returning node names that don't match registered names
> - Not storing completed workstream results in the state -- the synthesis node has nothing to combine
> - The loop condition checking the wrong field -- should check if all workstreams are processed
> - Forgetting to increment `current_workstream_idx` after each specialist execution
>
> **Skippable?** NO -- CRITICAL -- this is the core of the capstone. Without orchestration, it's just individual agents, not a multi-agent system. If students are really stuck, provide the graph structure code and let them focus on the node logic.

In [11]:
# Milestone 3 — SOLUTION: LangGraph Engagement Orchestration

class EngagementState(TypedDict):
    client_question: str
    plan: Dict
    completed_workstreams: List[Dict]
    current_workstream_idx: int
    deliverable: str

# Initialize the engagement team
em = EngagementManagerAgent()
specialists = {
    "strategy_analyst": StrategyAnalystAgent(),
    "financial_analyst": FinancialAnalystAgent(),
    "operations_expert": OperationsExpertAgent(),
    "industry_researcher": IndustryResearcherAgent()
}

def em_node(state: EngagementState) -> EngagementState:
    """Engagement Manager creates the workstream plan."""
    plan = em.decompose(state["client_question"])
    return {**state, "plan": plan, "completed_workstreams": [], "current_workstream_idx": 0}

def dispatcher_node(state: EngagementState) -> EngagementState:
    """Determine the next workstream to execute."""
    return state  # Just pass through — routing logic is in the conditional edge

def specialist_node(state: EngagementState) -> EngagementState:
    """Execute the current workstream with the assigned specialist."""
    workstreams = state["plan"]["workstreams"]
    idx = state["current_workstream_idx"]
    workstream = workstreams[idx]
    
    # Build context from completed workstreams (cross-workstream insight sharing)
    context = "\n\n".join([
        f"[{cw['worker'].upper()}] {cw['result'][:300]}" for cw in state["completed_workstreams"]
    ])
    
    # Execute with the assigned specialist
    specialist_type = workstream["assigned_specialist"]
    specialist = specialists.get(specialist_type, specialists["strategy_analyst"])
    result = specialist.execute(workstream["description"], context)
    
    completed = state["completed_workstreams"] + [{**result, "workstream_id": workstream["id"]}]
    return {**state, "completed_workstreams": completed, "current_workstream_idx": idx + 1}

def synthesis_node(state: EngagementState) -> EngagementState:
    """Synthesize all workstream outputs into a unified client deliverable."""
    parts = []
    for cw in state["completed_workstreams"]:
        specialist_title = cw["worker"].replace("_", " ").title()
        parts.append(f"--- [{specialist_title} Workstream] ---\n{cw['result']}")
    deliverable = "\n\n".join(parts)
    return {**state, "deliverable": deliverable}

def route_dispatcher(state: EngagementState) -> str:
    """Route to next specialist or synthesis based on remaining workstreams."""
    workstreams = state["plan"].get("workstreams", [])
    if state["current_workstream_idx"] < len(workstreams):
        return "specialist"
    return "synthesis"

# Build the engagement graph
graph = StateGraph(EngagementState)
graph.add_node("engagement_manager", em_node)
graph.add_node("dispatcher", dispatcher_node)
graph.add_node("specialist", specialist_node)
graph.add_node("synthesis", synthesis_node)

graph.set_entry_point("engagement_manager")
graph.add_edge("engagement_manager", "dispatcher")
graph.add_conditional_edges("dispatcher", route_dispatcher, {"specialist": "specialist", "synthesis": "synthesis"})
graph.add_edge("specialist", "dispatcher")
graph.add_edge("synthesis", END)

engagement_app = graph.compile()

# Test — Market entry strategy engagement
result = engagement_app.invoke({
    "client_question": (
        "A Fortune 500 industrial company wants to enter the Southeast Asian market "
        "for smart manufacturing solutions. Assess market attractiveness, competitive "
        "landscape, financial feasibility, and recommend an entry strategy."
    ),
    "plan": {}, "completed_workstreams": [], "current_workstream_idx": 0, "deliverable": ""
})

print(f"Completed {len(result['completed_workstreams'])} workstreams")
print(f"\nClient deliverable preview:\n{result['deliverable'][:500]}...")

Completed 4 workstreams

Client deliverable preview:
--- [Industry Researcher Workstream] ---
### Market Attractiveness Assessment for Smart Manufacturing Solutions in Southeast Asia

#### 1. Industry Landscape

**Market Size and Growth Rate:**
- The smart manufacturing market in Southeast Asia was valued at approximately **$10 billion in 2022** and is projected to grow at a compound annual growth rate (CAGR) of **15-20%** through 2027. This growth is driven by increasing investments in automation, IoT, and AI technologies.
- Key markets within So...


---
## Milestone 4: Partner Review & Quality Assurance Loop

Add a Partner-level quality review and self-correction loop to ensure deliverables meet McKinsey standards before going to the client.

> **Approach:** We add a Partner Review node that evaluates the synthesized deliverable on three dimensions: analytical rigor, actionability of recommendations, and strategic coherence. If the average score falls below the quality threshold (3.5/5), a revision node identifies specific weaknesses and rewrites. We track iteration count and cap retries at 2 — just as in a real engagement, there is a limit on revision cycles before the final deliverable must ship.
> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct This Exercise:** Give students 15-20 minutes. This adds a partner review loop -- the deliverable gets scored and revised if quality is below threshold. Suggest students implement the review node first, test it, then add the revision node and loop. Use a threshold of 3.5 average across 3 criteria. Prepare students: 'You'll demo this in Session 4 -- show the review scores improving across iterations.'
>
> **Common Student Mistakes:**
> - The partner review always giving high scores (>4) so the loop never triggers -- make the criteria strict
> - The revision node not knowing which criterion was weakest -- pass the scores and identify the lowest
> - No max revision limit -- the loop runs indefinitely if quality never reaches the threshold
> - Not tracking scores across iterations -- students can't show improvement in their demo
> - The revision rewriting the entire deliverable instead of targeting the weakest area
>
> **Skippable?** YES -- CAN SKIP if running behind schedule. Milestones 1-3 produce a working multi-agent system that is demonstrable. The partner review loop is a polish layer. If skipping, students can still demo a functional multi-agent engagement team. Encourage them to mention it as a 'future enhancement' in their presentation.

In [12]:
# Milestone 4 — SOLUTION: Partner Review & Quality Assurance Loop

class EnhancedEngagementState(TypedDict):
    client_question: str
    plan: Dict
    completed_workstreams: List[Dict]
    current_workstream_idx: int
    deliverable: str
    scores: List[Dict]
    revision_count: int

partner_llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def partner_review_node(state: EnhancedEngagementState) -> EnhancedEngagementState:
    """Partner reviews the deliverable for McKinsey-quality standards."""
    metrics = {}
    for metric, prompt in {
        "analytical_rigor": (
            f"Rate 1-5: Does this deliverable demonstrate rigorous, structured analysis with "
            f"clear hypotheses, supporting evidence, and logical reasoning?\n"
            f"Client question: {state['client_question']}\n"
            f"Deliverable: {state['deliverable'][:1000]}"
        ),
        "actionability": (
            f"Rate 1-5: Are the recommendations specific, actionable, and prioritized? "
            f"Could a CEO act on these immediately?\n"
            f"Deliverable: {state['deliverable'][:1000]}"
        ),
        "strategic_coherence": (
            f"Rate 1-5: Do the different workstream outputs tell a coherent story? "
            f"Are the insights synthesized into a unified strategic narrative?\n"
            f"Deliverable: {state['deliverable'][:1000]}"
        )
    }.items():
        resp = partner_llm.invoke([
            SystemMessage(content="You are a McKinsey Senior Partner reviewing a client deliverable. Return ONLY a number 1-5."),
            HumanMessage(content=prompt)
        ])
        try:
            metrics[metric] = int(resp.content.strip())
        except:
            metrics[metric] = 3
    
    metrics["average"] = round(sum(metrics.values()) / len(metrics), 2)
    scores = state.get("scores", []) + [metrics]
    print(f"  Partner Review (iteration {len(scores)}): {metrics}")
    return {**state, "scores": scores}

def revision_node(state: EnhancedEngagementState) -> EnhancedEngagementState:
    """Revise the deliverable based on Partner feedback — focus on the weakest dimension."""
    latest_scores = state["scores"][-1]
    
    # Identify weakest area
    weakest = min(
        ["analytical_rigor", "actionability", "strategic_coherence"],
        key=lambda k: latest_scores[k]
    )
    
    weakness_labels = {
        "analytical_rigor": "analytical rigor — strengthen hypotheses, add supporting evidence, sharpen logic",
        "actionability": "actionability — make recommendations more specific, prioritized, and implementation-ready",
        "strategic_coherence": "strategic coherence — better synthesize workstream insights into a unified narrative"
    }
    
    response = partner_llm.invoke([
        SystemMessage(content=f"""You are a McKinsey Engagement Manager revising a client deliverable based on Partner feedback.
The Partner flagged {weakness_labels[weakest]} as the key area for improvement.
Rewrite the deliverable to address this feedback while preserving the strong elements.
Maintain McKinsey-quality structure: situation, complication, resolution; clear exhibits; and a strong 'so what'."""),
        HumanMessage(content=f"Client question: {state['client_question']}\n\nCurrent deliverable:\n{state['deliverable']}")
    ])
    
    revision_count = state.get("revision_count", 0) + 1
    print(f"  Revised deliverable (focus: {weakest}, iteration {revision_count})")
    return {**state, "deliverable": response.content, "revision_count": revision_count}

def route_partner_review(state: EnhancedEngagementState) -> str:
    """Route based on Partner quality score and revision count."""
    scores = state.get("scores", [])
    revision_count = state.get("revision_count", 0)
    
    if not scores:
        return "end"
    
    latest = scores[-1]["average"]
    if latest >= 3.5 or revision_count >= 2:
        return "end"
    return "revise"

# Build enhanced engagement graph with Partner review loop
enhanced_graph = StateGraph(EnhancedEngagementState)
enhanced_graph.add_node("engagement_manager", em_node)
enhanced_graph.add_node("dispatcher", dispatcher_node)
enhanced_graph.add_node("specialist", specialist_node)
enhanced_graph.add_node("synthesis", synthesis_node)
enhanced_graph.add_node("partner_review", partner_review_node)
enhanced_graph.add_node("reviser", revision_node)

enhanced_graph.set_entry_point("engagement_manager")
enhanced_graph.add_edge("engagement_manager", "dispatcher")
enhanced_graph.add_conditional_edges("dispatcher", route_dispatcher, {"specialist": "specialist", "synthesis": "synthesis"})
enhanced_graph.add_edge("specialist", "dispatcher")
enhanced_graph.add_edge("synthesis", "partner_review")
enhanced_graph.add_conditional_edges("partner_review", route_partner_review, {"revise": "reviser", "end": END})
enhanced_graph.add_edge("reviser", "partner_review")

enhanced_engagement_app = enhanced_graph.compile()

# Test — Full PE due diligence engagement with Partner review
result = enhanced_engagement_app.invoke({
    "client_question": (
        "Our PE client is evaluating a $500M acquisition of a healthcare IT platform company. "
        "Conduct a comprehensive due diligence covering market dynamics, competitive positioning, "
        "financial health, and operational scalability. Provide a go/no-go recommendation "
        "with key risks, value creation levers, and a 100-day integration plan."
    ),
    "plan": {}, "completed_workstreams": [], "current_workstream_idx": 0,
    "deliverable": "", "scores": [], "revision_count": 0
})

print(f"\nCompleted {len(result['completed_workstreams'])} workstreams, {result['revision_count']} revisions")
print(f"Partner review scores: {[s['average'] for s in result['scores']]}")
print(f"\nFinal client deliverable:\n{result['deliverable'][:600]}...")

  Partner Review (iteration 1): {'analytical_rigor': 4, 'actionability': 3, 'strategic_coherence': 4, 'average': 3.67}

Completed 4 workstreams, 0 revisions
Partner review scores: [3.67]

Final client deliverable:
--- [Industry Researcher Workstream] ---
### Healthcare IT Market Dynamics Analysis

#### 1. Industry Landscape

**Market Size and Growth Rate:**
- The global healthcare IT market was valued at approximately **$250 billion in 2022** and is projected to grow at a CAGR of **15%**, reaching around **$500 billion by 2027**. This growth is driven by increasing demand for digital health solutions, the need for improved patient care, and the rising prevalence of chronic diseases.

**Key Trends:**
- **Telehealth Expansion:** The COVID-19 pandemic accelerated the adoption of telehealth services, which ...


---
### Milestone 5: Cross-Agent Memory and Context Sharing

Add a shared knowledge store that agents can read from and write to. When the Financial Analyst discovers a key metric, the Strategy Analyst should be able to reference it without re-deriving it.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Start by motivating the problem: 'What happens when Agent A discovers something Agent B needs?' Show the SharedKnowledgeStore pattern first, then have students wire it into their existing agents. Test with a scenario where one agent's finding should influence another.
>
> **Common Student Mistakes:**
> - Not handling concurrent writes to the shared store safely
> - Storing raw LLM output instead of structured facts in the knowledge store
> - Creating circular dependencies where agents keep reading each other's outputs
> - Forgetting to namespace or tag facts by source agent
>
> **Skippable?** YES
> Cross-agent memory is an advanced orchestration pattern. If running low on time, demonstrate the instructor solution and discuss the architectural pattern. Core multi-agent workflow from Milestones 1-4 is complete.
> ---


In [13]:
# Milestone 5 — SOLUTION: Cross-Agent Memory and Context Sharing

class SharedKnowledgeStore:
    """Shared memory store for cross-agent context during an engagement."""
    
    def __init__(self):
        self.facts = []       # List of {"agent": ..., "fact": ..., "category": ...}
        self.metrics = {}     # key -> value (e.g., "revenue_growth" -> "15%")
        self.assumptions = [] # Shared assumptions across workstreams
    
    def add_fact(self, agent_name, fact, category="general"):
        self.facts.append({"agent": agent_name, "fact": fact, "category": category})
    
    def add_metric(self, key, value, source_agent):
        self.metrics[key] = {"value": value, "source": source_agent}
    
    def add_assumption(self, assumption, agent_name):
        self.assumptions.append({"assumption": assumption, "from": agent_name})
    
    def get_context_for_agent(self, requesting_agent):
        """Get all shared knowledge from OTHER agents for cross-pollination."""
        other_facts = [f for f in self.facts if f["agent"] != requesting_agent]
        context_parts = []
        if other_facts:
            context_parts.append("Findings from other workstreams:")
            for f in other_facts[-5:]:
                context_parts.append(f"  - [{f['agent']}] {f['fact']}")
        if self.metrics:
            context_parts.append("\nShared metrics:")
            for k, v in list(self.metrics.items())[-5:]:
                context_parts.append(f"  - {k}: {v['value']} (from {v['source']})")
        if self.assumptions:
            context_parts.append("\nShared assumptions:")
            for a in self.assumptions[-3:]:
                context_parts.append(f"  - {a['assumption']} (from {a['from']})")
        return "\n".join(context_parts) if context_parts else "No shared context yet."
    
    def summary(self):
        return {
            "total_facts": len(self.facts),
            "total_metrics": len(self.metrics),
            "total_assumptions": len(self.assumptions),
            "contributing_agents": list(set(f["agent"] for f in self.facts))
        }


class ContextAwareSpecialist:
    """A specialist agent that reads from and writes to the shared knowledge store."""
    
    def __init__(self, name, role_description, knowledge_store):
        self.name = name
        self.role = role_description
        self.store = knowledge_store
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
    
    def analyze(self, workstream):
        """Analyze a workstream using shared context from other agents."""
        shared_context = self.store.get_context_for_agent(self.name)
        
        response = self.llm.invoke([
            SystemMessage(content=(
                f"You are {self.name}, a {self.role} at McKinsey. "
                f"Analyze the workstream below. Use shared context from other agents when relevant. "
                f"After your analysis, list 1-2 KEY FINDINGS that other agents should know about. "
                f"Format key findings as: KEY_FINDING: <finding>"
            )),
            HumanMessage(content=(
                f"Shared context:\n{shared_context}\n\n"
                f"Your workstream:\n{json.dumps(workstream, indent=2)}"
            ))
        ])
        
        analysis = response.content
        
        # Extract and store key findings
        for line in analysis.split("\n"):
            if "KEY_FINDING:" in line:
                finding = line.split("KEY_FINDING:")[-1].strip()
                self.store.add_fact(self.name, finding, category=workstream.get("focus", "general"))
        
        return {"agent": self.name, "analysis": analysis}


# Test
store = SharedKnowledgeStore()

fin_agent = ContextAwareSpecialist("Financial Analyst", "financial due diligence specialist", store)
strategy_agent = ContextAwareSpecialist("Strategy Analyst", "competitive strategy specialist", store)

# Financial analyst goes first
fin_result = fin_agent.analyze({"focus": "financial_health", "hypothesis": "Target has strong revenue but declining margins"})
print(f"Financial Analyst:\n{fin_result['analysis'][:300]}...\n")

# Strategy analyst uses financial findings
strat_result = strategy_agent.analyze({"focus": "competitive_position", "hypothesis": "Target has strong market position in healthcare IT"})
print(f"Strategy Analyst:\n{strat_result['analysis'][:300]}...\n")

print(f"Shared store summary: {store.summary()}")


Financial Analyst:
To analyze the workstream focused on the financial health of the target company, we will assess the hypothesis that the target has strong revenue but declining margins. 

1. **Revenue Analysis**: 
   - Examine the revenue growth trends over the past few years. Strong revenue typically indicates a so...

Strategy Analyst:
In analyzing the competitive position of the target company in the healthcare IT sector, it is essential to consider both the findings from the financial analyst and the broader market dynamics. The target's strong revenue growth indicates a robust demand for its products or services, which is a pos...

Shared store summary: {'total_facts': 4, 'total_metrics': 0, 'total_assumptions': 0, 'contributing_agents': ['Financial Analyst', 'Strategy Analyst']}


---
### Milestone 6: Engagement Analytics and Reporting Dashboard

Build an analytics layer that tracks agent performance, token costs, workstream timing, and quality scores across engagements. This enables engagement managers to monitor AI-assisted engagement efficiency.

> ---
> **INSTRUCTOR GUIDANCE**
>
> **How to Conduct:** Present this as the 'observability layer' for AI engagements. Have students instrument their existing pipeline with event logging, then aggregate stats. Show a simple dashboard output comparing agent performance metrics.
>
> **Common Student Mistakes:**
> - Logging too much detail causing performance overhead
> - Not tracking token costs per agent, missing the cost attribution
> - Building complex visualizations instead of focusing on the data collection
> - Forgetting to include timestamps for timing analysis
>
> **Skippable?** YES
> Analytics dashboard is a stretch goal. If time is tight, show the instructor solution and discuss what metrics matter for production AI systems. The core multi-agent pipeline is already complete.
> ---


In [14]:
# Milestone 6 — SOLUTION: Engagement Analytics and Reporting Dashboard

class EngagementAnalytics:
    """Tracks and reports on multi-agent engagement performance."""
    
    def __init__(self):
        self.events = []  # List of engagement events
    
    def log_event(self, event_type, agent_name, data=None):
        """Log an engagement event."""
        self.events.append({
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "agent": agent_name,
            "data": data or {}
        })
    
    def log_workstream(self, agent_name, workstream_title, duration_sec, quality_score=None):
        self.log_event("workstream_completed", agent_name, {
            "workstream": workstream_title,
            "duration_sec": duration_sec,
            "quality_score": quality_score
        })
    
    def log_review(self, iteration, scores, passed):
        self.log_event("partner_review", "Partner", {
            "iteration": iteration,
            "scores": scores,
            "passed": passed
        })
    
    def dashboard(self):
        """Generate an engagement performance dashboard."""
        workstream_events = [e for e in self.events if e["event_type"] == "workstream_completed"]
        review_events = [e for e in self.events if e["event_type"] == "partner_review"]
        
        # Agent performance
        agent_stats = defaultdict(lambda: {"workstreams": 0, "total_time": 0, "scores": []})
        for e in workstream_events:
            agent = e["agent"]
            agent_stats[agent]["workstreams"] += 1
            agent_stats[agent]["total_time"] += e["data"].get("duration_sec", 0)
            if e["data"].get("quality_score"):
                agent_stats[agent]["scores"].append(e["data"]["quality_score"])
        
        for agent, stats in agent_stats.items():
            stats["avg_time_sec"] = round(stats["total_time"] / max(stats["workstreams"], 1), 1)
            stats["avg_quality"] = round(np.mean(stats["scores"]), 2) if stats["scores"] else None
        
        # Review stats
        review_stats = {
            "total_reviews": len(review_events),
            "passed_first_try": sum(1 for e in review_events if e["data"].get("iteration") == 1 and e["data"].get("passed")),
            "required_revision": sum(1 for e in review_events if not e["data"].get("passed"))
        }
        
        return {
            "total_events": len(self.events),
            "total_workstreams": len(workstream_events),
            "agent_performance": dict(agent_stats),
            "review_stats": review_stats,
            "total_engagement_time_sec": sum(e["data"].get("duration_sec", 0) for e in workstream_events)
        }


# Test
analytics = EngagementAnalytics()

# Simulate engagement events
analytics.log_workstream("Strategy Analyst", "Market positioning analysis", 45, quality_score=4)
analytics.log_workstream("Financial Analyst", "Revenue and margin assessment", 60, quality_score=5)
analytics.log_workstream("Operations Expert", "Operational scalability review", 55, quality_score=3)
analytics.log_workstream("Industry Researcher", "Healthcare IT market trends", 40, quality_score=4)
analytics.log_review(iteration=1, scores={"analytical_rigor": 3, "actionability": 4, "strategic_coherence": 3}, passed=False)
analytics.log_review(iteration=2, scores={"analytical_rigor": 4, "actionability": 5, "strategic_coherence": 4}, passed=True)

dashboard = analytics.dashboard()
print("=== Engagement Analytics Dashboard ===")
print(f"Total workstreams: {dashboard['total_workstreams']}")
print(f"Total engagement time: {dashboard['total_engagement_time_sec']}s")
print(f"\nAgent Performance:")
for agent, stats in dashboard["agent_performance"].items():
    print(f"  {agent}: {stats['workstreams']} workstreams, avg {stats['avg_time_sec']}s, quality={stats['avg_quality']}")
print(f"\nReview Stats: {dashboard['review_stats']}")


=== Engagement Analytics Dashboard ===
Total workstreams: 4
Total engagement time: 200s

Agent Performance:
  Strategy Analyst: 1 workstreams, avg 45.0s, quality=4.0
  Financial Analyst: 1 workstreams, avg 60.0s, quality=5.0
  Operations Expert: 1 workstreams, avg 55.0s, quality=3.0
  Industry Researcher: 1 workstreams, avg 40.0s, quality=4.0

Review Stats: {'total_reviews': 2, 'passed_first_try': 0, 'required_revision': 1}


---
## Capstone Summary

Students built a **McKinsey Multi-Agent Engagement Team** — a production-ready multi-agent system modeled after real consulting engagement structure:

1. **Engagement Manager (Supervisor)** -- Decomposes client questions into workstreams with hypotheses, assigns specialists, and manages dependencies
2. **Consulting Specialists (Workers)** -- Strategy Analyst, Financial Analyst, Operations Expert, and Industry Researcher — each with domain-specific analytical frameworks
3. **Engagement Orchestration** -- LangGraph workflow with conditional routing, cross-workstream context sharing, and deliverable synthesis
4. **Partner Review (Self-Correction)** -- Quality evaluation on analytical rigor, actionability, and strategic coherence — with automated revision until McKinsey standards are met

**Key consulting scenarios demonstrated:** PE due diligence, market entry strategy, digital transformation assessment, cost optimization programs.

**Up Next:** Session 4 -- Cross-track presentations, AI governance, and closing.